# Carga de los 3 datasets 


In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_ventas = pd.read_csv("../../data\S11_PD_Diaz_DatasetTransformado.csv")
df_bodega = pd.read_csv("../../data/bodega_categorias_novamarket.csv")
df_crm = pd.read_csv("../../data\crm_clientes_novamarket.csv")

<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:7: SyntaxWarning: invalid escape sequence '\c'
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:7: SyntaxWarning: invalid escape sequence '\c'
C:\Users\diego\AppData\Local\Temp\ipykernel_27704\2077152207.py:5: SyntaxWarning: invalid escape sequence '\S'
  df_ventas = pd.read_csv("../../data\S11_PD_Diaz_DatasetTransformado.csv")
C:\Users\diego\AppData\Local\Temp\ipykernel_27704\2077152207.py:7: SyntaxWarning: invalid escape sequence '\c'
  df_crm = pd.read_csv("../../data\crm_clientes_novamarket.csv")


In [29]:
print(df_ventas.info())

<class 'pandas.DataFrame'>
RangeIndex: 570976 entries, 0 to 570975
Data columns (total 56 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id_pedido                    570976 non-null  str    
 1   id_cliente                   570976 non-null  str    
 2   fecha_pedido                 550688 non-null  str    
 3   ciudad_tienda                570976 non-null  str    
 4   codigo_postal                570976 non-null  int64  
 5   producto                     570976 non-null  str    
 6   precio_unitario              570976 non-null  float64
 7   unidades_vendidas            570976 non-null  float64
 8   monto_compra                 570976 non-null  float64
 9   edad_cliente                 570976 non-null  float64
 10  correo_cliente               477145 non-null  str    
 11  nivel_satisfaccion           570976 non-null  str    
 12  nivel_lealtad                570976 non-null  str    
 13  comentario

In [30]:
print(df_bodega.info())

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   categoria            7 non-null      str  
 1   proveedor_principal  7 non-null      str  
 2   bodega_asignada      7 non-null      str  
 3   dias_reposicion      7 non-null      int64
 4   stock_minimo         7 non-null      int64
dtypes: int64(2), str(3)
memory usage: 412.0 bytes
None


In [31]:
print(df_crm.info())

<class 'pandas.DataFrame'>
RangeIndex: 411840 entries, 0 to 411839
Data columns (total 6 columns):
 #   Column                     Non-Null Count   Dtype
---  ------                     --------------   -----
 0   email_cliente              411840 non-null  str  
 1   ciudad_residencia          411840 non-null  str  
 2   fecha_inscripcion_club     411840 non-null  str  
 3   puntos_fidelidad_actuales  411840 non-null  int64
 4   segmento_crm               411840 non-null  str  
 5   acepta_promociones         411840 non-null  str  
dtypes: int64(1), str(5)
memory usage: 18.9 MB
None


# 1. inspeccion

## 1. nombres de columnas 

In [32]:
print(df_ventas.columns.tolist())

['id_pedido', 'id_cliente', 'fecha_pedido', 'ciudad_tienda', 'codigo_postal', 'producto', 'precio_unitario', 'unidades_vendidas', 'monto_compra', 'edad_cliente', 'correo_cliente', 'nivel_satisfaccion', 'nivel_lealtad', 'comentario_cliente', 'peso_pedido_kg', 'fecha_actualizacion_stock', 'fuga_cliente', 'correo_cliente_faltante', 'comentario_cliente_faltante', 'es_venta_corporativa', 'edad_cliente_era_nulo', 'precio_unitario_era_nulo', 'unidades_vendidas_era_nulo', 'edad_cliente_escalada', 'monto_compra_escalada', 'precio_unitario_escalada', 'unidades_vendidas_escalada', 'nivel_lealtad_cod', 'cat_Belleza', 'cat_Deportes', 'cat_Electrónica', 'cat_Hogar', 'cat_Juguetería', 'cat_Moda', 'pago_Efectivo', 'pago_Paypal', 'pago_Tarjeta', 'pago_Transferencia', 'canal_App', 'canal_Marketplace', 'canal_Tienda', 'canal_Web', 'ciudad_tienda_cod', 'codigo_postal_agrupado', 'mes_pedido', 'trimestre_pedido', 'dia_semana_pedido', 'es_fin_semana_pedido', 'longitud_comentario', 'num_palabras_comentario', 

In [33]:
df_bodega.columns.tolist()

['categoria',
 'proveedor_principal',
 'bodega_asignada',
 'dias_reposicion',
 'stock_minimo']

In [34]:
df_crm.columns.tolist()

['email_cliente',
 'ciudad_residencia',
 'fecha_inscripcion_club',
 'puntos_fidelidad_actuales',
 'segmento_crm',
 'acepta_promociones']

### Conflictos de nombre detectados

**Sinónimos reales** (mismo concepto, distinto nombre; se deben armonizar para el join):

- `correo_cliente` (ventas) ↔ `email_cliente` (CRM): correo electrónico del cliente. Es la llave del join ventas–CRM. Nombre unificado propuesto: `correo_cliente`.
- `cat_*` (ventas, dummies one-hot: `cat_Belleza`, `cat_Deportes`, `cat_Electrónica`, `cat_Hogar`, `cat_Juguetería`, `cat_Moda`) ↔ `categoria` (bodega): categoría del producto. En ventas está codificada en columnas dummy, no en una sola columna. Hay que reconstruir `categoria` antes del join con bodega.

**Falsos amigos** (nombres parecidos, semántica distinta; NO se deben tratar como equivalentes):

- `ciudad_tienda` (ventas) vs `ciudad_residencia` (CRM): ciudad donde se hizo la compra vs ciudad donde vive el cliente. Pueden diferir legítimamente.
- `nivel_lealtad` (ventas) vs `segmento_crm` / `puntos_fidelidad_actuales` (CRM): ambos describen lealtad, pero con escalas y orígenes distintos. Hay que verificar si se solapan antes de combinarlos.

**Llave ausente:**

- `id_cliente` (ventas) no tiene equivalente en el CRM, que solo identifica clientes por email. Por eso el join depende de `correo_cliente`, que tiene nulos MNAR.

In [ ]:
df_ventas["correo_cliente"] = df_ventas["correo_cliente"].str.strip().str.lower()
df_crm["email_cliente"]     = df_crm["email_cliente"].str.strip().str.lower()
df_bodega["categoria"]      = df_bodega["categoria"].str.strip().str.lower()

Se aplicó `.str.strip().str.lower()` a los correos como medida preventiva. Se verificó que ni ventas ni CRM contienen mayúsculas, y que los matches son idénticos antes y después (379.354), por lo que la normalización no altera el resultado del join. Se conserva para garantizar robustez ante datos futuros.

para hacer el join es necesario re construir la columna de categoria en ventas que se modifico en s9

In [36]:
cols_cat = [c for c in df_ventas.columns if c.startswith("cat_")]

# Cuántas dummies activas hay por fila (debería ser 1 en todas)
print(df_ventas[cols_cat].sum(axis=1).value_counts())

# Reconstrucción: la dummy activa da el nombre de la categoría
activa = df_ventas[cols_cat].astype(bool)
df_ventas["categoria"] = (
    activa.idxmax(axis=1).str.replace("cat_", "", regex=False)
    .where(activa.any(axis=1))   # NaN si ninguna dummy está activa
    .str.strip().str.lower()
)

print(df_ventas["categoria"].value_counts(dropna=False))

1    570976
Name: count, dtype: int64
categoria
electrónica    95606
belleza        95519
deportes       95380
hogar          95027
moda           94723
juguetería     94721
Name: count, dtype: int64


In [37]:
print(sorted(df_bodega["categoria"].unique()))
print(sorted(df_ventas["categoria"].unique()))
print("En ventas pero no en bodega:", set(df_ventas["categoria"]) - set(df_bodega["categoria"]))

['deportes', 'electrónica', 'hogar', 'juguetería', 'mascotas', 'moda', 'papelería']
['belleza', 'deportes', 'electrónica', 'hogar', 'juguetería', 'moda']
En ventas pero no en bodega: {'belleza'}


# 2. merge 

In [39]:
df_bodega = df_bodega.rename(columns={"categoria": "categoria_producto"})
df_ventas = df_ventas.rename(columns={"categoria": "categoria_producto"})

In [40]:
filas_antes = df_ventas.shape[0]
print(filas_antes)

570976


In [41]:
df_paso1 = pd.merge(df_ventas, df_bodega, on="categoria_producto", how="left", validate="m:1")

In [42]:
print(filas_antes, df_paso1.shape[0], filas_antes == df_paso1.shape[0])

570976 570976 True


In [43]:
df_crm = df_crm.rename(columns={"email_cliente": "correo_cliente"})

In [45]:
df_integrado = pd.merge(df_paso1, df_crm, on="correo_cliente", how="left", validate="m:1")

In [46]:
print(filas_antes, df_integrado.shape[0], filas_antes == df_integrado.shape[0])
print(df_integrado.columns.tolist())

570976 570976 True
['id_pedido', 'id_cliente', 'fecha_pedido', 'ciudad_tienda', 'codigo_postal', 'producto', 'precio_unitario', 'unidades_vendidas', 'monto_compra', 'edad_cliente', 'correo_cliente', 'nivel_satisfaccion', 'nivel_lealtad', 'comentario_cliente', 'peso_pedido_kg', 'fecha_actualizacion_stock', 'fuga_cliente', 'correo_cliente_faltante', 'comentario_cliente_faltante', 'es_venta_corporativa', 'edad_cliente_era_nulo', 'precio_unitario_era_nulo', 'unidades_vendidas_era_nulo', 'edad_cliente_escalada', 'monto_compra_escalada', 'precio_unitario_escalada', 'unidades_vendidas_escalada', 'nivel_lealtad_cod', 'cat_Belleza', 'cat_Deportes', 'cat_Electrónica', 'cat_Hogar', 'cat_Juguetería', 'cat_Moda', 'pago_Efectivo', 'pago_Paypal', 'pago_Tarjeta', 'pago_Transferencia', 'canal_App', 'canal_Marketplace', 'canal_Tienda', 'canal_Web', 'ciudad_tienda_cod', 'codigo_postal_agrupado', 'mes_pedido', 'trimestre_pedido', 'dia_semana_pedido', 'es_fin_semana_pedido', 'longitud_comentario', 'num_pal

### pedidos sin categoría en bodega

In [50]:
print(df_integrado["proveedor_principal"].isnull().sum())

95519


### pedidos de clientes fuera del CRM

In [51]:
print(df_integrado["ciudad_residencia"].isnull().sum())

191622


### verificar que los números sean confiables

In [52]:
print(df_bodega["proveedor_principal"].isnull().sum(), df_crm["ciudad_residencia"].isnull().sum())

0 0


## Registros sin coincidencia

- **95.519 pedidos sin categoría en bodega** (≈16,7 %): corresponden a la categoría `belleza`, que existe en ventas pero no está registrada en el catálogo de bodega. Es esperable porque el catálogo tiene cobertura incompleta, no por un error del join.
- **191.622 pedidos de clientes ausentes en el CRM** (≈33,6 %): el CRM solo contiene clientes inscritos en el club de fidelización, por lo que quien compró sin inscribirse, o sin registrar correo, no tiene registro. Es esperable porque el `left join` conserva todos los pedidos aunque el cliente no esté en el CRM.

In [53]:
print(pd.crosstab(df_integrado["ciudad_residencia"].isnull(), df_integrado["correo_cliente_faltante"]))

correo_cliente_faltante       0      1
ciudad_residencia                     
False                    379354      0
True                      97791  93831


## Conflicto de valor: ciudad_tienda vs. ciudad_residencia

Aunque ambas columnas contienen nombres de ciudades, no describen lo mismo: `ciudad_tienda` (ventas) registra dónde ocurrió la compra, y `ciudad_residencia` (CRM) registra dónde vive el cliente según su inscripción al club. Es normal que difieran, por ejemplo cuando alguien compra en otra ciudad por trabajo, viaje o por canal online, así que la diferencia no es un error.

Por eso no se elimina ni se sobrescribe ninguna de las dos: borrar una perdería información, y sobrescribir una con la otra introduciría un dato falso. Además, `ciudad_residencia` solo existe para los clientes presentes en el CRM (los 191.622 pedidos sin coincidencia quedan con nulo), de modo que usarla para rellenar `ciudad_tienda` mezclaría conceptos distintos. Ambas columnas se conservan tal como están en el dataset final, y su comparación puede servir más adelante como señal analítica (compra fuera de la ciudad de residencia).

In [54]:
df_integrado.to_csv("../../data/S12_PD_Diaz_DatasetIntegrado.csv", index=False)

In [55]:
df_check = pd.read_csv("../../data/S12_PD_Diaz_DatasetIntegrado.csv")
print(df_integrado.shape, df_check.shape, df_integrado.shape == df_check.shape)

(570976, 66) (570976, 66) True
